# Phase 4.6 — ttm GPU compatibility
Installation and execution are pending until this notebook succeeds on a real GPU. Use a fresh Colab GPU session for each model. No test evaluation, tuning, adapters, or GitHub write credentials. Run cells in order. Model work runs in an isolated interpreter, never the notebook's package environment.

In [ ]:
# 1. GPU/runtime: do not import notebook-kernel torch.
import hashlib
import json
import os
import subprocess
import sys
from pathlib import Path

assert (3, 11) <= sys.version_info[:2] <= (3, 12), "Use a Python 3.11/3.12 Colab runtime"
subprocess.run(["nvidia-smi"], check=True)
print("kernel Python", sys.version.split()[0])

In [ ]:
# 2. Pin the PUBLISHED phase-4.6 repository commit shown in the handoff.
PROJECT_COMMIT = input("Paste the full phase-4.6 commit SHA: ").strip()
assert len(PROJECT_COMMIT) == 40 and all(c in "0123456789abcdef" for c in PROJECT_COMMIT)
ROOT = Path("/content/tsfm-zero-few-shot-crossover")
URL = "https://github.com/Han-Youseung/tsfm-zero-few-shot-crossover.git"
if not ROOT.exists():
    subprocess.run(["git", "clone", URL, str(ROOT)], check=True)
assert (
    subprocess.check_output(
        ["git", "-C", str(ROOT), "remote", "get-url", "origin"], text=True
    ).strip()
    == URL
)
assert not subprocess.check_output(
    ["git", "-C", str(ROOT), "status", "--porcelain"], text=True
).strip()
subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", PROJECT_COMMIT], check=True)
os.chdir(ROOT)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip() == PROJECT_COMMIT
print("Immutable execution commit:", PROJECT_COMMIT)
# Never git pull during this run.

In [ ]:
# 3. Optional persistent destination. Drive mounting is a USER action.
USE_DRIVE = (
    False  # Set True before starting if you want completed results to survive disconnection.
)
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    RESULT_DIR = Path("/content/drive/MyDrive/tsfm-gpu-gate") / PROJECT_COMMIT / "ttm"
else:
    RESULT_DIR = ROOT / "results/raw/gpu" / PROJECT_COMMIT / "ttm"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print("Temporary disk is NOT persistent; download results promptly unless using Drive.")

In [ ]:
import uuid

INSTALL_FILE = RESULT_DIR / f"installation-{uuid.uuid4().hex}.json"
# 4. Separate model venv. Never apply a CPU dependency snapshot.
ENV = Path("/content/venv-ttm-phase46")
PY = ENV / "bin/python"
if not PY.exists():
    subprocess.run(["apt-get", "update"], check=True)
    subprocess.run(["apt-get", "install", "-y", "python3.12-venv"], check=True)
    subprocess.run([sys.executable, "-m", "venv", str(ENV)], check=True)
setup = {
    "kind": "gpu_installation",
    "model_family": "ttm",
    "execution_commit": PROJECT_COMMIT,
    "status": "installation_pending",
}
try:
    subprocess.run([str(PY), "-m", "pip", "install", "--upgrade", "pip"], check=True)
    subprocess.run([str(PY), "-m", "pip", "install", "-r", "requirements/ttm-gpu.txt"], check=True)
    subprocess.run([str(PY), "-m", "pip", "check"], check=True)
except subprocess.CalledProcessError:
    setup["status"] = "failed"
    setup["category"] = "installation_dependency"
    raise
finally:
    temp = INSTALL_FILE.with_suffix(".json.tmp")
    temp.write_text(json.dumps(setup, indent=2))
    temp.replace(INSTALL_FILE)

In [ ]:
# 5. The SAME interpreter installed packages and executes every probe.
# Kernel remains separate by design: do not import model libraries into it.
check_code = """
import json
from tsfm_crossover.models.gpu_smoke import environment
import torch
assert torch.cuda.is_available() and torch.version.cuda
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
print(json.dumps(environment('ttm', 'fe7a35697723e2a2f5246ae979474bfc554e26c0'), indent=2))
"""
try:
    subprocess.run([str(PY), "-c", check_code], check=True)
    setup["status"] = "verified"
except subprocess.CalledProcessError:
    setup["status"] = "failed"
    setup["category"] = "installation_or_runtime"
    raise
finally:
    temp = INSTALL_FILE.with_suffix(".json.tmp")
    temp.write_text(json.dumps(setup, indent=2))
    temp.replace(INSTALL_FILE)

In [ ]:
# 6. Official raw ETTh1 only; no benchmark bundle or test windows.
import urllib.request

DATA = ROOT / "data/official_raw/ETTh1.csv"
DATA.parent.mkdir(parents=True, exist_ok=True)
if not DATA.exists():
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/1d16c8f4f943005d613b5bc962e9eeb06058cf07/ETT-small/ETTh1.csv",
        DATA,
    )
assert (
    hashlib.sha256(DATA.read_bytes()).hexdigest()
    == "f18de3ad269cef59bb07b5438d79bb3042d3be49bdeecf01c1cd6d29695ee066"
)
print("Official ETTh1 fingerprint verified")

## 7–11. Ordered condition checks in the existing probe
Each horizon runs in a fresh subprocess: immutable revision/config hash → synthetic CUDA smoke → validation Zero-Shot/repeat/channel check → fresh pretrained model → train full-parameter step → new model and optimizer checkpoint/RNG restore → validation comparison. FP32 is mandatory; AMP is explicitly not_run. No automatic precision policy. Each condition is atomically saved; completed matching conditions resume. Failures remain recorded.

In [ ]:
# Execute four mandatory FP32 conditions. Continue collecting other conditions after a failure.
CONDITIONS = []
for horizon in (96, 192, 336, 720):
    destination = RESULT_DIR / f"ttm-h{horizon}-s1.json"
    command = [
        str(PY),
        "scripts/probe_ttm_cpu.py",
        "--gpu-gate",
        "--horizon",
        str(horizon),
        "--num-samples",
        "1",
        "--expected-commit",
        PROJECT_COMMIT,
        "--data",
        str(DATA),
        "--cache-dir",
        str(ROOT / ".cache/ttm-gpu"),
        "--output",
        str(destination),
    ]
    completed = subprocess.run(command, check=False)
    CONDITIONS.append(destination)
    print(horizon, "exit code:", completed.returncode)
    if destination.exists():
        print(json.loads(destination.read_text())["status"])
# Any failed/missing condition blocks the mandatory gate.

In [ ]:
# 12. Validate small JSON; both-model production gate requires both sessions' results.
existing = [str(p) for p in CONDITIONS if p.exists()]
assert len(existing) == len(CONDITIONS), (
    "Missing results: inspect process/install failure; do not mark passed"
)
subprocess.run(
    [
        str(PY),
        "-m",
        "tsfm_crossover.models.gpu_gate",
        "--expected-commit",
        PROJECT_COMMIT,
        *existing,
    ],
    check=True,
)
print("Scope: ETTh1, 7 channels, batch size 1. No full-dataset feasibility claim.")

In [ ]:
# 13. Download only small JSON summaries (including failed attempts). No model files.
import zipfile

from google.colab import files

archive = Path("/content/ttm-gpu-results.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(RESULT_DIR.glob("*.json")):
        assert path.stat().st_size < 2_000_000, "Unexpected large result"
        bundle.write(path, arcname=path.name)
files.download(str(archive))
print("Download now. Only a user-mounted Drive destination survives temporary runtime loss.")

## Local return
Extract the ZIP into ignored results/raw/gpu-return/. Validate condition JSON only (exclude installation-*.json), using the SAME execution commit entered above:

python -m tsfm_crossover.models.gpu_gate --expected-commit <SHA> --import-results <condition.json> ...

This imports immutable evidence into results/manifests/models/gpu_runs/<SHA>/, never overwrites CPU manifests, and prints the combined gate. Review all eight mandatory results together before updating selection status. Installation failures/runtime interruptions are not model rejection. Do not automatically implement adapters or freeze the research protocol.